# 38 — gen-3.6 screen · LANZAR EN EL POD

**Handoff 2026-08-12 (legokna → quien lance), medido en el pod el 13-ago.** Todo está construido y
validado; falta correr.

**Corre en el POD**, no en UNAM: necesita los datos del challenge, que no pueden estar en la
máquina universitaria hasta que haya acuerdo escrito (`context/UNAM_SERVER.md`).

## Qué está ya validado (en UNAM, cero datos del challenge, cero GPU de entrenamiento)

| | dónde |
|---|---|
| Tubería cargar→LoRA→2 pasos→merge, 6/6 | `RESULTS_smoke_unsloth.json` |
| La ruta de eval lee un merge de Unsloth | `RESULTS_eval_path.json` |
| La ruta de entrega FP8 | `RESULTS_fp8_llmcompressor.json` |
| Por qué NO es ms-swift | `RESULTS_viability_v2.json` |

## Ya medido en el pod con datos reales (`RESULTS_smoke_pod_27b.json`)

- ✅ **23,5 s/it sostenido** (el paso 1 fue 230,6 s: compilación y warm-up) × 900,9 pasos ⇒
  **una época ≈ 5,9 h**. ⚠️ Es UNA medición sostenida; trátala como indicativa.
- 🔴 **Pico 52,64 GiB ⇒ hace falta una GPU de ≥80 GB.** No cabe en un L40S de 48.
- 🔴 **`df` MIENTE en los volúmenes de RunPod** — reporta el clúster MooseFS, no la cuota (~640 GB).
  Comprueba el espacio de verdad; el 13-ago un merge murió por `Disk quota exceeded`.

## El sujeto: **`Qwen/Qwen3.6-27B`** — decidido 2026-08-13

Necesita el paso FP8 para servir (celda 5), ya validado. La alternativa medida y descartada era
`Qwen/Qwen3.5-9B` (~18 GB bf16, sin paso FP8, lectura más limpia por ser casi paritario con
nuestro 8B). Cambiar de uno a otro sigue siendo **una línea** en la celda 4.

## 1 · Entorno — usar la ruta DOCUMENTADA de Unsloth

⚠️ En UNAM usamos conda + pip plano y nos costó un `torchvision::nms` roto. Su doc dice
`uv pip install unsloth --torch-backend=auto` **en venv**, y *"do NOT use this if you have Conda"*.
En el pod, seguir su ruta.

In [ ]:
!python -m venv /workspace/envs/unsloth && \
 /workspace/envs/unsloth/bin/pip install -q uv && \
 /workspace/envs/unsloth/bin/uv pip install --python /workspace/envs/unsloth/bin/python \
     unsloth --torch-backend=auto

# verificación: torch, GPU y que unsloth importe
!/workspace/envs/unsloth/bin/python -c "\
import unsloth, torch, transformers; \
print('unsloth', unsloth.__version__, '| transformers', transformers.__version__); \
print('torch', torch.__version__, '| gpus', torch.cuda.device_count(), \
      '| cap', torch.cuda.get_device_capability(0)); \
from unsloth import FastVisionModel; print('FastVisionModel OK')"

## 2 · Comprobar rutas antes de nada

La guarda de sha256 caza un `train.jsonl` equivocado al instante, pero mejor verlo antes.

In [ ]:
from pathlib import Path
import hashlib

TRAIN = Path('/workspace/repo/experiments/18-count-aug/runs/18_count_aug_v1/train.jsonl')
print('existe:', TRAIN.exists())
if TRAIN.exists():
    print('filas :', sum(1 for _ in open(TRAIN)))
    print('sha256:', hashlib.sha256(TRAIN.read_bytes()).hexdigest())
    print('esperado: 180e28f0325674197d52706beeabd846851bdd2875264b5c3505e0debfbd8e8b')
    import json
    r = json.loads(open(TRAIN).readline())
    img = Path(r['images'][0])
    print('primer frame existe:', img.exists(), '→', img)

## 3 · SMOKE con datos reales — 2 pasos

🔴 **No saltar.** Confirma que el 27B carga, que el `train.jsonl` real convierte, y **da el `s/it`**.

In [ ]:
import sys; sys.path.insert(0, '/workspace/repo/experiments/38-gen36-ft-screen')
from _models.unsloth_sft import Config, main

res = main(Config(run_name='38_smoke', smoke=True, smoke_steps=2, smoke_rows=32))
print('\nLoRA por parte:', res['lora_by_part'])   # merger 0 es ESPERADO, no un fallo
print('pico VRAM      :', res['peak_vram_gib'], 'GiB')
print('segundos       :', res['train_secs'], '→ mira el s/it antes de la celda 4')

## 4 · EL BRAZO — 1 época

Lánzalo dentro de **`tmux`** (`tmux new -s ft38`) para que una desconexión no lo mate.

**Si muere, se reanuda: vuelve a correr ESTA MISMA celda, sin cambiar nada.** `main()` busca el
último checkpoint en `runs/38_qwen36_27b_v1/ckpt/` y continúa desde su paso; si no hay ninguno,
arranca limpio. `RESULTS_train.json` registra `resumed_from`, así que un número que salga de un run
reanudado se puede identificar como tal.

- Guarda cada **100 pasos** (~39 min de los 900,9 pasos), conservando los 2 últimos.
- **Dónde va sin abrir nada:** `cat runs/38_qwen36_27b_v1/HEARTBEAT.json` → paso, %, loss, s/it y
  ETA en horas. El log completo queda en `runs/38_qwen36_27b_v1/train.log` (fichero, no scrollback).

🔴 **La GPU:** pico medido **52,64 GiB** ⇒ hace falta **≥80 GB** (A100/H100). No cabe en un L40S 48.

In [ ]:
cfg = Config(
    model='Qwen/Qwen3.6-27B',      # ← alternativa: 'Qwen/Qwen3.5-9B' (sin FP8 al servir)
    run_name='38_qwen36_27b_v1',
    num_train_epochs=1,
    # receta A2 trasladada; los defaults ya la llevan
    learning_rate=2e-4, lora_rank=8, lora_alpha=32,
    gradient_accumulation_steps=16, seed=42,
    max_pixels=1280*720,           # 🔴 Unsloth por defecto usa 512
)
res = main(cfg)
res

## 5 · Cuantizar a FP8 (sólo si el sujeto es el 27B)

**Data-free**: no necesita datos, así que puede correr donde sea.
⚠️ **Entorno SEPARADO** — `llmcompressor` pide `transformers>=5.9.0` y Unsloth capa en `<=5.5.0`.

In [ ]:
!python -m venv /workspace/envs/quant && /workspace/envs/quant/bin/pip install -q llmcompressor
!/workspace/envs/quant/bin/python /workspace/repo/experiments/38-gen36-ft-screen/_tools/quantize_fp8.py \
    /workspace/repo/experiments/38-gen36-ft-screen/runs/38_qwen36_27b_v1/merged \
    /workspace/repo/experiments/38-gen36-ft-screen/runs/38_qwen36_27b_v1/merged_fp8

## 6 · Evaluar y comparar

`frame.run.run_baseline` con `cfg.engine_factory` apuntando a
`experiments/23-backbone-screen/_tools/screen_engine.py` (ya arreglado para transformers 5.x).

### 🔴 LA VARA ESTÁ EN DISPUTA — resolver ANTES de leer el eval (2026-08-13)

Esta celda decía *"batir a **A2 ep3**: proxy local 0.6104, `bucket_mean` 0.6496"*. **Ninguno de esos
dos números aparece en los resultados de A2**, y contradicen el diseño escrito en el README.

| fuente | qué dice |
|---|---|
| README de este rung, §"Why this rung exists" | el test es *una época leída contra el checkpoint de **época 1** de A2* |
| `02-lora-sft/runs/02_lora_sft_v1/ep1_full/stratified.json` | A2 **ep1** `bucket_mean` = **0.5282** |
| `02-lora-sft/RESULTS.csv` + `stratified.json` | A2 **ckpt elegido** (época 2 de 3) `bucket_mean` = **0.5486** |
| `0.6496` | aparece en el repo **solo** como `ci_low` dentro del `stratified.json` de A2 |
| `0.6104` | **no aparece en ninguna parte** |

⇒ Sospecha fuerte de que la vara se transcribió de una cota del intervalo de confianza. Y la
diferencia decide el veredicto: contra **0.5282** (lo que el README diseñó) el listón está 0.02 por
debajo de lo que A2 sacó con su checkpoint elegido; contra un inventado **0.6496** casi cualquier
resultado sale NO-GO.

**No lo resuelvo yo**: hace falta que el equipo confirme de dónde salió el 0.6104. Mientras tanto,
la comparación honesta y trazable es **A2 ep1 = 0.5282**, que es la que el README nombra y la única
que sale de `frame.metrics` (RULES: `bucket_mean` es la headline, nunca re-derivada a mano).

⚠️ Deflactar antes de leerlo como leaderboard: el `bucket_mean` local sobreestima **+0.121** y el
orden de buckets está invertido en `obj_OOD` ([[local-eval-vs-judge-calibration]]). El signo
sobrevive 8/8; la magnitud no.